# 2. Data Wrangling and Feature Engineering
Reads the raw JSON files from notebook 01, constructs one row per historical launch, joins rocket and launchpad information, and exports a cleaned CSV.

In [4]:

from pathlib import Path
import json
import requests
import pandas as pd
import numpy as np
import time

# 1. Create the data folder
DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# 2. API endpoints
API_BASES = [
    "https://api.spacexdata.com/v4",
    "https://gateway.pipeworx.io/spacex/v4",
]

# 3. Download JSON data
def fetch_spacex_data(endpoint):
    errors = []

    for base in API_BASES:
        url = f"{base}/{endpoint}"

        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            data = response.json()

            if not isinstance(data, list) or not data:
                raise ValueError("API returned empty or unexpected data")

            print(f"Downloaded {endpoint}: {len(data)} records")
            return data

        except Exception as e:
            errors.append(f"{url}: {e}")
            print(f"Failed: {url}")

    raise RuntimeError(
        f"Could not download {endpoint}.\n" + "\n".join(errors)
    )

# 4. Download required datasets
datasets = {
    "spacex_launches.json": "launches",
    "spacex_rockets.json": "rockets",
}

for filename, endpoint in datasets.items():
    records = fetch_spacex_data(endpoint)
    output_path = DATA_DIR / filename

    output_path.write_text(
        json.dumps(records, indent=2),
        encoding="utf-8"
    )

    # Also save a CSV copy
    pd.json_normalize(records).to_csv(
        DATA_DIR / filename.replace(".json", ".csv"),
        index=False
    )

# 5. Verify saved files
print("\nFiles saved:")
for file in sorted(DATA_DIR.iterdir()):
    print(file.name, "-", file.stat().st_size, "bytes")

print("\nLaunches:", len(json.loads(
    (DATA_DIR / "spacex_launches.json").read_text()
)))
print("Rockets:", len(json.loads(
    (DATA_DIR / "spacex_rockets.json").read_text()
)))

print("\nData collection complete. Now rerun Notebook 02.")


Failed: https://api.spacexdata.com/v4/launches
Downloaded launches: 101 records
Failed: https://api.spacexdata.com/v4/rockets
Downloaded rockets: 13 records

Files saved:
spacex_launches.csv - 10116 bytes
spacex_launches.json - 17143 bytes
spacex_rockets.csv - 2668 bytes
spacex_rockets.json - 5462 bytes

Launches: 101
Rockets: 13

Data collection complete. Now rerun Notebook 02.


In [6]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

# Load launches from the saved JSON file
DATA_DIR = Path("/content/data")
launch_file = DATA_DIR / "spacex_launches.json"

if not launch_file.exists():
    raise FileNotFoundError(
        "Launch data not found. Run the data collection cell first."
    )

with open(launch_file, "r", encoding="utf-8") as f:
    launches = json.load(f)

# Convert to DataFrame
launch_df = pd.json_normalize(launches)

# Display requested columns
columns = ["flight_number", "name", "date_utc", "success", "upcoming"]
available_columns = [col for col in columns if col in launch_df.columns]

display(launch_df[available_columns].head())
print("Total launch records:", len(launch_df))

,name,date_utc,success
0,Falcon Heavy | NROL-97,2026-10-02T03:54:00Z,True
1,Falcon 9 Block 5 | Transporter 18 (Dedicated S...,2026-10-01T18:32:00Z,True
2,Falcon 9 Block 5 | Crew-13,2026-10-01T15:10:06Z,True
3,Starship | Starlink Group 31-1 (Starship Fligh...,2026-09-28T12:48:59Z,True
4,Falcon 9 Block 5 | USSF-385,2026-09-26T14:00:54Z,True


Total launch records: 101


In [10]:

import pandas as pd
import numpy as np

def first_item(value):
    if isinstance(value, list):
        return value[0] if value else None
    return value if isinstance(value, str) and value else None

def get_column(data, column, default=None):
    if column in data.columns:
        return data[column]
    return pd.Series([default] * len(data), index=data.index)

# Build cleaned launch dataset
df = pd.DataFrame({
    "flight_number": get_column(launch_df, "flight_number"),
    "name": get_column(launch_df, "name"),
    "date_utc": pd.to_datetime(
        get_column(launch_df, "date_utc"),
        errors="coerce",
        utc=True
    ),
    "success": get_column(launch_df, "success"),
    "upcoming": get_column(launch_df, "upcoming"),
    "rocket_id": get_column(launch_df, "rocket").apply(first_item),
    "launchpad_id": get_column(launch_df, "launchpad").apply(first_item),
    "payload_ids": get_column(launch_df, "payloads", []),
    "cores": get_column(launch_df, "cores", [])
})

# Feature engineering
df["year"] = df["date_utc"].dt.year
df["month"] = df["date_utc"].dt.month

df["payload_count"] = df["payload_ids"].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

df["core_reuse_count"] = df["cores"].apply(
    lambda xs: sum(
        int(c.get("reuse_count") or 0)
        for c in xs if isinstance(c, dict)
    ) if isinstance(xs, list) else 0
)

df["core_landing_success"] = df["cores"].apply(
    lambda xs: any(
        c.get("landing_success") is True
        for c in xs if isinstance(c, dict)
    ) if isinstance(xs, list) else False
)

print("Cleaned dataset shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

display(df.head())


Cleaned dataset shape: (101, 14)

Missing values:
flight_number           101
name                      0
date_utc                  0
success                   1
upcoming                101
rocket_id               100
launchpad_id            101
payload_ids               0
cores                     0
year                      0
month                     0
payload_count             0
core_reuse_count          0
core_landing_success      0
dtype: int64


,flight_number,name,date_utc,success,upcoming,rocket_id,launchpad_id,payload_ids,cores,year,month,payload_count,core_reuse_count,core_landing_success
0,None,Falcon Heavy | NROL-97,2026-10-02 03:54:00+00:00,True,None,None,None,[],[],2026,10,0,0,False
1,None,Falcon 9 Block 5 | Transporter 18 (Dedicated S...,2026-10-01 18:32:00+00:00,True,None,None,None,[],[],2026,10,0,0,False
2,None,Falcon 9 Block 5 | Crew-13,2026-10-01 15:10:06+00:00,True,None,None,None,[],[],2026,10,0,0,False
3,None,Starship | Starlink Group 31-1 (Starship Fligh...,2026-09-28 12:48:59+00:00,True,None,None,None,[],[],2026,9,0,0,False
4,None,Falcon 9 Block 5 | USSF-385,2026-09-26 14:00:54+00:00,True,None,None,None,[],[],2026,9,0,0,False


In [12]:
print("Source columns:", launch_df.columns.tolist())
display(launch_df.head(2))

Source columns: ['name', 'status', 'success', 'date_utc', 'pad', 'rocket', 'details', 'links.article', 'links.webcast', 'links.wikipedia']


,name,status,success,date_utc,pad,rocket,details,links.article,links.webcast,links.wikipedia
0,Falcon Heavy | NROL-97,Launch Successful,True,2026-10-02T03:54:00Z,NaN,NaN,NaN,NaN,NaN,NaN
1,Falcon 9 Block 5 | Transporter 18 (Dedicated S...,Launch Successful,True,2026-10-01T18:32:00Z,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:

from pathlib import Path
import json
import pandas as pd
import numpy as np

DATA_DIR = Path("/content/data")

def load_records(filename):
    path = DATA_DIR / filename
    if not path.exists():
        print(f"Warning: {filename} not found.")
        return []
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)
    return records if isinstance(records, list) else []

# Load source datasets
rockets = load_records("spacex_rockets.json")
launchpads = load_records("spacex_launchpads.json")

print("Rockets:", len(rockets))
print("Launchpads:", len(launchpads))

# Build rocket lookup
rocket_df = pd.json_normalize(rockets)

if not rocket_df.empty:
    if "id" not in rocket_df.columns and "_id" in rocket_df.columns:
        rocket_df["id"] = rocket_df["_id"]

    rocket_cols = [
        c for c in [
            "id", "name", "type", "success_rate_pct",
            "cost_per_launch", "active", "stages", "boosters"
        ] if c in rocket_df.columns
    ]

    rocket_lookup = rocket_df[rocket_cols].rename(columns={
        "id": "rocket_id",
        "name": "rocket_name",
        "type": "rocket_type",
        "success_rate_pct": "rocket_success_rate_pct",
        "cost_per_launch": "rocket_cost_per_launch"
    })
else:
    rocket_lookup = pd.DataFrame(columns=["rocket_id"])

# Build launchpad lookup
pad_df = pd.json_normalize(launchpads)

if not pad_df.empty:
    if "id" not in pad_df.columns and "_id" in pad_df.columns:
        pad_df["id"] = pad_df["_id"]

    pad_cols = [
        c for c in [
            "id", "name", "full_name", "locality", "region",
            "latitude", "longitude", "launch_attempts",
            "launch_successes", "status"
        ] if c in pad_df.columns
    ]

    pad_lookup = pad_df[pad_cols].rename(columns={
        "id": "launchpad_id",
        "name": "launchpad_name",
        "full_name": "launchpad_full_name"
    })
else:
    pad_lookup = pd.DataFrame(columns=["launchpad_id"])

# Merge only when the corresponding ID exists
if "rocket_id" in df.columns and "rocket_id" in rocket_lookup.columns:
    df = df.merge(rocket_lookup, on="rocket_id", how="left")
else:
    print("Rocket merge skipped: rocket_id is unavailable.")

if "launchpad_id" in df.columns and "launchpad_id" in pad_lookup.columns:
    df = df.merge(pad_lookup, on="launchpad_id", how="left")
else:
    print("Launchpad merge skipped: launchpad_id is unavailable.")

print("\nJoined dataset shape:", df.shape)
display(df.head(3))


Rockets: 13
Launchpads: 0
Rocket merge skipped: rocket_id is unavailable.

Joined dataset shape: (101, 14)


,flight_number,name,date_utc,success,upcoming,rocket_id,launchpad_id,payload_ids,cores,year,month,payload_count,core_reuse_count,core_landing_success
0,None,Falcon Heavy | NROL-97,2026-10-02 03:54:00+00:00,True,None,None,None,[],[],2026,10,0,0,False
1,None,Falcon 9 Block 5 | Transporter 18 (Dedicated S...,2026-10-01 18:32:00+00:00,True,None,None,None,[],[],2026,10,0,0,False
2,None,Falcon 9 Block 5 | Crew-13,2026-10-01 15:10:06+00:00,True,None,None,None,[],[],2026,10,0,0,False


In [15]:

from pathlib import Path
import json
import pandas as pd
import numpy as np

DATA_DIR = Path("/content/data")

# Load payload data safely
payload_file = DATA_DIR / "spacex_payloads.json"

if payload_file.exists():
    with open(payload_file, "r", encoding="utf-8") as f:
        payloads = json.load(f)
    if not isinstance(payloads, list):
        payloads = []
else:
    payloads = []
    print("Warning: spacex_payloads.json not found.")

payload_df = pd.json_normalize(payloads)

# Calculate payload mass if the expected columns exist
id_col = next(
    (c for c in ["id", "_id"] if c in payload_df.columns),
    None
)
mass_col = "mass_kg" if "mass_kg" in payload_df.columns else None

if id_col and mass_col:
    payload_mass_lookup = payload_df.set_index(id_col)[mass_col].to_dict()

    df["payload_mass_kg"] = df["payload_ids"].apply(
        lambda ids: sum(
            float(payload_mass_lookup.get(pid) or 0)
            for pid in ids
        ) if isinstance(ids, list) else np.nan
    )
else:
    df["payload_mass_kg"] = np.nan
    print(
        "Payload mass unavailable. Columns found:",
        payload_df.columns.tolist()
    )

# Exclude upcoming launches
if "upcoming" in df.columns:
    df = df[df["upcoming"] != True].copy()

# Normalize success labels
df["success"] = pd.to_numeric(df["success"], errors="coerce")
df["success"] = df["success"].where(df["success"].isin([0, 1]))
df["success_label"] = df["success"].map({
    1.0: "Success",
    0.0: "Failure"
})

# Report results
print("Payload records:", len(payloads))
print("Cleaned dataset shape:", df.shape)
print("\nMissing-value fraction:")
display(
    df.isna()
      .mean()
      .sort_values(ascending=False)
      .head(15)
      .rename("missing_fraction")
)

display(df.head())


Payload mass unavailable. Columns found: []
Payload records: 0
Cleaned dataset shape: (101, 16)

Missing-value fraction:


,missing_fraction
flight_number,1.000000
upcoming,1.000000
launchpad_id,1.000000
payload_mass_kg,1.000000
rocket_id,0.990099
success,0.009901
success_label,0.009901
name,0.000000
payload_ids,0.000000
date_utc,0.000000


,flight_number,name,date_utc,success,upcoming,rocket_id,launchpad_id,payload_ids,cores,year,month,payload_count,core_reuse_count,core_landing_success,payload_mass_kg,success_label
0,None,Falcon Heavy | NROL-97,2026-10-02 03:54:00+00:00,1.0,None,None,None,[],[],2026,10,0,0,False,NaN,Success
1,None,Falcon 9 Block 5 | Transporter 18 (Dedicated S...,2026-10-01 18:32:00+00:00,1.0,None,None,None,[],[],2026,10,0,0,False,NaN,Success
2,None,Falcon 9 Block 5 | Crew-13,2026-10-01 15:10:06+00:00,1.0,None,None,None,[],[],2026,10,0,0,False,NaN,Success
3,None,Starship | Starlink Group 31-1 (Starship Fligh...,2026-09-28 12:48:59+00:00,1.0,None,None,None,[],[],2026,9,0,0,False,NaN,Success
4,None,Falcon 9 Block 5 | USSF-385,2026-09-26 14:00:54+00:00,1.0,None,None,None,[],[],2026,9,0,0,False,NaN,Success


In [16]:
# Drop large nested fields from the analytic CSV; preserve raw JSON separately.
for col in ["payload_ids","cores","details","upcoming"]:
    if col in df.columns:
        df = df.drop(columns=col)
out = DATA_DIR / "spacex_launches_cleaned.csv"
df.to_csv(out, index=False)
print("Saved:", out, "| rows:", len(df), "| columns:", len(df.columns))
display(df.head())

Saved: /content/data/spacex_launches_cleaned.csv | rows: 101 | columns: 13


,flight_number,name,date_utc,success,rocket_id,launchpad_id,year,month,payload_count,core_reuse_count,core_landing_success,payload_mass_kg,success_label
0,None,Falcon Heavy | NROL-97,2026-10-02 03:54:00+00:00,1.0,None,None,2026,10,0,0,False,NaN,Success
1,None,Falcon 9 Block 5 | Transporter 18 (Dedicated S...,2026-10-01 18:32:00+00:00,1.0,None,None,2026,10,0,0,False,NaN,Success
2,None,Falcon 9 Block 5 | Crew-13,2026-10-01 15:10:06+00:00,1.0,None,None,2026,10,0,0,False,NaN,Success
3,None,Starship | Starlink Group 31-1 (Starship Fligh...,2026-09-28 12:48:59+00:00,1.0,None,None,2026,9,0,0,False,NaN,Success
4,None,Falcon 9 Block 5 | USSF-385,2026-09-26 14:00:54+00:00,1.0,None,None,2026,9,0,0,False,NaN,Success


## Data-quality notes
Review missingness before modelling. Unknown outcomes are left missing and should not be treated as failures. The exact dataset can change because the API is live. Record the rows excluded as upcoming and the missing-value patterns from your run.